# 02. Feature Engineering

## Purpose
Create the customer features that later analysis needs to compare groups: age group, income group and tenure group.



## 1. Load the cleaned datasets
We read the cleaned files produced by `01_data_cleaning`, not the raw files, so every feature is built on data that has already been checked.

In [ ]:
import pandas as pd
import numpy as np

# cleaned files created in 01_data_cleaning
customers = pd.read_csv('../data/cleaned/customers_clean.csv')
offers = pd.read_csv('../data/cleaned/offers_clean.csv')
events = pd.read_csv('../data/cleaned/events_clean.csv')

## 2. Demographic features
Raw age, income and join date are too detailed to compare groups of customers. We turn them into a few readable groups so we can later ask questions like "do 25-34 year olds complete more offers than 55-64 year olds?".

New columns on the customer table:
- `age_group`
- `income_group`
- `tenure_days` and `tenure_group` (how long each customer has been a member)

The group boundaries are analyst judgement, not something the data dictates. They are chosen so each group is meaningful for the business and has enough customers to compare.

### 2.1 Age and income groups
`pd.cut` sorts each value into a range. Ranges are **right-inclusive**: with `age_bins = [17, 24, ...]` the first group is ages 18 to 24 (greater than 17 and up to 24), which is why the bins start at 17 and not 18.

In [ ]:
# ---- Age groups ----
# bins are the range edges (7 edges make 6 ranges); labels name each range in order
age_bins = [17, 24, 34, 44, 54, 64, 200]
age_labels = [
    '18-24 Young Adults', '25-34 Early Career', '35-44 Young Families',
    '45-54 Mature Professionals', '55-64 Pre-Retirement', '65+ Retirees'
]
customers['age_group'] = pd.cut(customers['age'], bins=age_bins, labels=age_labels)

# ---- Income groups ----
# income in the data runs from $30,000 to $120,000, so the first range effectively starts at 30k
income_bins = [0, 50000, 75000, 100000, 200000]
income_labels = [
    '30-50k Lower-Middle', '50-75k Middle', '75-100k Upper-Middle', '100-120k Affluent'
]
customers['income_group'] = pd.cut(customers['income'], bins=income_bins, labels=income_labels)

In [ ]:
# Tenure feature: how long each customer has been a member.
# became_member_on was saved to CSV as text, so it comes back as 'object' (text), not a date.
print(customers['became_member_on'].dtype)

# convert to a real date so we can subtract dates from each other
customers['became_member_on'] = pd.to_datetime(customers['became_member_on'])

In [ ]:
# check: should now show datetime64[ns]
print(customers['became_member_on'].dtype)

In [ ]:
# Reference date = the latest join date in the data (not today's real date).
# The dataset is a 30-day snapshot, so tenure is measured "as of" the end of that period.
reference_date = customers['became_member_on'].max()
customers['tenure_days'] = (reference_date - customers['became_member_on']).dt.days

# -1 as the first edge so a customer who joined on the reference date (0 days) is included
tenure_bins = [-1, 365, 1095, 1825, 10000]
tenure_labels = ['New (<1yr)', 'Established (1-3yr)', 'Loyal (3-5yr)', 'Veteran (5yr+)']
customers['tenure_group'] = pd.cut(customers['tenure_days'], bins=tenure_bins, labels=tenure_labels)

In [ ]:
# guards: tenure must vary, and join dates must be real dates (not numbers read as 1970)
assert customers['became_member_on'].dt.year.min() >= 2013
assert customers['tenure_group'].nunique() > 1

customers.to_csv('../data/cleaned/customers_features.csv', index=False)
print("customers_features:", customers.shape)

### 2.2 Summary of the new features

| Column | What it is | Groups |
|---|---|---|
| `age_group` | age band | 18-24, 25-34, 35-44, 45-54, 55-64, 65+ |
| `income_group` | annual income band | 30-50k, 50-75k, 75-100k, 100-120k |
| `tenure_days` | days between joining and the latest join date in the data | number |
| `tenure_group` | membership length band | New (<1yr), Established (1-3yr), Loyal (3-5yr), Veteran (5yr+) |

### Assumptions
- Group boundaries are judgement calls. Different boundaries could change how patterns look, so results should be read as "by these groups", not as exact thresholds.
- Tenure is measured to the latest join date in the data, not to today, because the data is a 30-day snapshot.